# Saving data to JSON files

This notebook saves all of the Perplexity data to JSON files named after the scenario the data is pulled from.

These JSON files are then converted to the `graph_json` format that POGG requires in order to build a graph.

In other words, this file just exists to do the query on the prolog data and store it so it doesn't need to be queried again.

In [29]:
# DIRECTORIES
original_prolog_dir = "/Users/lizcconrad/Documents/PhD/POGG/Adventure/NLTK"

mode = "test"
data_json_directory = "/Users/lizcconrad/Documents/PhD/POGG/data_handling/perplexity/{}/data_jsons".format(mode)

In [30]:
import json
import os
import pathlib
import re
from swiplserver import PrologMQI

# set working directory to the directory with the Perplexity data
os.chdir(original_prolog_dir)

if mode == "development":
    # prolog filenames
    prolog_filenames = [
        'Debug_POGG_Tutorial.pl',
        'Debug_POGG_HealTheCar.pl',
        'Debug_POGG_HealTheTrees.pl',
    ]
else:
    prolog_filenames = [
        'Debug_POGG_AtomicCity.pl',
        'Debug_POGG_baby.pl',
        'Debug_POGG_kidneykwest.pl',
        'Debug_POGG_Scenario.pl',
        'Debug_POGG_HealTheCave.pl',
        'Debug_POGG_HealTheFlashback.pl',
        'Debug_POGG_HealTheLake.pl',
    ]


result_jsons = {}

for filename in prolog_filenames:
    scenario_name = re.match('Debug_POGG_(.*).pl', filename).group(1)

    # open Prolog thread
    with PrologMQI() as mqi:
        with mqi.create_thread() as prolog_thread:
            # consult Debug_POGG_[SCENARIO].pl
            prolog_thread.query("['{}']".format(filename))

            # get all entities with property and relationship data
            result = prolog_thread.query("getEntityData(Entity, Properties, Relationships).")

    result_jsons[scenario_name] = result

# set working directory to the data_handling directory
pathlib.Path(data_json_directory).mkdir(parents=True, exist_ok=True)
os.chdir(data_json_directory)

Prolog: Warning: /Users/lizcconrad/Documents/PhD/POGG/Adventure/NLTK/Debug_POGG_AtomicCity.pl:76:
Prolog: Warning:    Singleton variables: [Properties,Nears]
Prolog: **** AtomicCity ****
Prolog: Warning: /Users/lizcconrad/Documents/PhD/POGG/Adventure/NLTK/Debug_POGG_baby.pl:76:
Prolog: Warning:    Singleton variables: [Properties,Nears]
Prolog: **** baby ****
Prolog: Warning: /Users/lizcconrad/Documents/PhD/POGG/Adventure/NLTK/Debug_POGG_kidneykwest.pl:76:
Prolog: Warning:    Singleton variables: [Properties,Nears]
Prolog: **** kidneykwest ****
Prolog: Warning: /Users/lizcconrad/Documents/PhD/POGG/Adventure/NLTK/Debug_POGG_Scenario.pl:76:
Prolog: Warning:    Singleton variables: [Properties,Nears]
Prolog: **** Scenario ****
Prolog: Warning: /Users/lizcconrad/Documents/PhD/POGG/Adventure/NLTK/Debug_POGG_HealTheCave.pl:76:
Prolog: Warning:    Singleton variables: [Properties,Nears]
Prolog: **** Heal_TheCave ****
Prolog: Warning: /Users/lizcconrad/Documents/PhD/POGG/Adventure/NLTK/Debug_P

In [31]:
for scenario_name in result_jsons.keys():
    with open(os.path.join(data_json_directory, "{}_raw_data.json".format(scenario_name)), "w") as json_file:
        json_file.write(json.dumps(result_jsons[scenario_name], indent=4))